In [0]:
# Load cleaned Silver tables

villages_df = spark.table("workspace.agri_intelligence.silver_villages")

farmers_df = spark.table("workspace.agri_intelligence.silver_farmers")

products_df = spark.table("workspace.agri_intelligence.silver_products")

bills_df = spark.table("workspace.agri_intelligence.silver_bills")

bill_items_df = spark.table("workspace.agri_intelligence.silver_bill_items")

print("All Silver tables loaded successfully.")

In [0]:
from pyspark.sql.functions import col, coalesce, lower, trim

# Give each table a short name to avoid confusion during joins

b = bills_df.alias("b")
i = bill_items_df.alias("i")
f = farmers_df.alias("f")
p = products_df.alias("p")
v = villages_df.alias("v")


# Join bills with their individual items

gold_sales_df = b.join(
    i,
    col("b.bill_id") == col("i.bill_id"),
    "inner"
)


# Add farmer details

gold_sales_df = gold_sales_df.join(
    f,
    col("b.farmer_id") == col("f.farmer_id"),
    "left"
)


# Add product details

gold_sales_df = gold_sales_df.join(
    p,
    col("i.product_id") == col("p.product_id"),
    "left"
)


# Get the correct village details

gold_sales_df = gold_sales_df.join(
    v,
    coalesce(
        col("f.village_id"),
        col("b.village_id_snapshot")
    ) == col("v.village_id"),
    "left"
)


# Select the columns required for business reporting

gold_sales_df = gold_sales_df.select(
    col("b.bill_id"),
    col("b.bill_number"),
    col("b.bill_date"),
    col("b.farmer_id"),

    coalesce(
        col("f.farmer_name"),
        col("b.customer_name_snapshot")
    ).alias("farmer_name"),

    coalesce(
        col("v.village_name"),
        col("b.village_name_snapshot"),
        col("f.village_name")
    ).alias("village_name"),

    coalesce(
        col("v.normalised_village_name"),
        lower(trim(col("b.village_name_snapshot")))
    ).alias("normalised_village_name"),

    col("i.bill_item_id"),
    col("i.product_id"),

    col("p.product_group"),
    col("p.crop"),
    col("p.company_name"),
    col("p.brand_name"),
    col("p.variety_name"),

    coalesce(
        col("p.product_display_name"),
        col("i.product_display_name")
    ).alias("product_display_name"),

    col("p.pack_size_value"),
    col("p.pack_size_unit"),

    col("i.quantity_packs"),
    col("i.unit_rate_inr"),
    col("i.line_total_inr")
)

print("Gold sales dataset created successfully.")

In [0]:
# Check the Gold sales records

gold_sales_df.printSchema()

gold_sales_df.show(10, truncate=False)

print("Total sales records:", gold_sales_df.count())

In [0]:
# Save the combined sales data as a Gold Delta table

gold_sales_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.agri_intelligence.gold_sales")

print("Gold sales table saved successfully.")

In [0]:
# Check the saved Gold table

gold_sales_df = spark.table("workspace.agri_intelligence.gold_sales")

print("Total sales records:", gold_sales_df.count())

In [0]:
from pyspark.sql.functions import (
    col, sum as spark_sum, countDistinct,
    min as spark_min, max as spark_max, when
)

# Keep identified farmers for the Farmer Memory report

identified_sales = gold_sales_df.filter(
    col("farmer_id").isNotNull()
)


# Calculate each farmer's purchase history

gold_farmer_summary = identified_sales.groupBy(
    "farmer_id",
    "farmer_name",
    "village_name"
).agg(
    countDistinct("bill_id").alias("total_bills"),
    spark_sum("line_total_inr").alias("total_spent_inr"),
    spark_min("bill_date").alias("first_purchase_date"),
    spark_max("bill_date").alias("last_purchase_date"),
    countDistinct("variety_name").alias("different_varieties")
)


# Identify repeat customers

gold_farmer_summary = gold_farmer_summary.withColumn(
    "customer_type",
    when(col("total_bills") > 1, "Repeat Customer")
    .otherwise("One-time Customer")
)


# Display farmers with the highest spending

gold_farmer_summary.orderBy(
    col("total_spent_inr").desc()
).show(10, truncate=False)

In [0]:
# Save farmer purchase summaries as a Gold Delta table

gold_farmer_summary.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.agri_intelligence.gold_farmer_summary")

print("Farmer Memory table saved successfully.")

In [0]:
# Check the number of identified farmers in the summary

print("Farmer summary records:", gold_farmer_summary.count())

gold_farmer_summary.show(10, truncate=False)

In [0]:
# Load the existing Gold Sales table

gold_sales_df = spark.table(
    "workspace.agri_intelligence.gold_sales"
)

In [0]:
from pyspark.sql.functions import (
    col, sum as spark_sum, countDistinct,
    round, avg
)

# Calculate sales performance for identified villages only

gold_village_summary = gold_sales_df.filter(
    col("village_name").isNotNull()
).groupBy(
    "normalised_village_name",
    "village_name"
).agg(
    spark_sum("line_total_inr").alias("total_revenue_inr"),
    countDistinct("farmer_id").alias("unique_farmers"),
    countDistinct("bill_id").alias("total_bills"),
    round(avg("line_total_inr"), 2).alias("average_item_value_inr")
)

# Display villages with the highest revenue

gold_village_summary.orderBy(
    col("total_revenue_inr").desc()
).show(10, truncate=False)

# Save the updated Gold table

gold_village_summary.write.mode("overwrite").saveAsTable(
    "workspace.agri_intelligence.gold_village_summary"
)

In [0]:
village_tables = [
    "gold_village_summary",
    "gold_village_average_bill",
    "gold_village_top_products",
    "gold_monthly_revenue"
]

for table_name in village_tables:
    row_count = spark.table(
        f"workspace.agri_intelligence.{table_name}"
    ).count()

    print(f"{table_name}: {row_count} rows")

In [0]:
from pyspark.sql.functions import col, avg, round

# Get one record per bill

village_bill_summary = gold_sales_df.select(
    "bill_id",
    "village_name",
    "normalised_village_name"
).distinct()

# Add bill totals from the Silver bills table

village_bill_summary = village_bill_summary.join(
    bills_df.select("bill_id", "bill_total_inr"),
    "bill_id",
    "inner"
)

# Calculate average bill value for each village

village_average_bill = village_bill_summary.groupBy(
    "normalised_village_name",
    "village_name"
).agg(
    round(avg("bill_total_inr"), 2).alias("average_bill_value_inr")
)

village_average_bill.show(10, truncate=False)

In [0]:
# Find sales records without a village

gold_sales_df.filter(
    col("village_name").isNull()
).select(
    "bill_id",
    "farmer_id",
    "farmer_name",
    "village_name"
).distinct().show(20, False)

In [0]:
# Calculate average bill value for identified villages only

village_average_bill = village_bill_summary.filter(
    col("village_name").isNotNull()
).groupBy(
    "normalised_village_name",
    "village_name"
).agg(
    round(avg("bill_total_inr"), 2).alias("average_bill_value_inr")
)

village_average_bill.show(10, False)

In [0]:
village_average_bill.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.agri_intelligence.gold_village_average_bill"
)

In [0]:
from pyspark.sql.functions import sum as spark_sum

# Total packs sold for each variety in each village
village_product_sales = gold_sales_df.filter(
    col("village_name").isNotNull()
).groupBy(
    "normalised_village_name",
    "village_name",
    "product_display_name",
    "crop"
).agg(
    spark_sum("quantity_packs").alias("total_packs_sold"),
    spark_sum("line_total_inr").alias("total_revenue_inr")
)

# Show the best-selling varieties
village_product_sales.orderBy(
    col("village_name"),
    col("total_packs_sold").desc()
).show(100, False)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Rank products by packs sold within each village
village_window = Window.partitionBy("village_name").orderBy(
    col("total_packs_sold").desc(),
    col("total_revenue_inr").desc()
)

top_products_by_village = village_product_sales.withColumn(
    "product_rank",
    row_number().over(village_window)
).filter(
    col("product_rank") <= 5
)

# Display the top 5 products in each village
top_products_by_village.orderBy(
    "village_name",
    "product_rank"
).show(100, False)

In [0]:
# Save the top 5 products for each village
top_products_by_village.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.agri_intelligence.gold_village_top_products"
)

In [0]:
from pyspark.sql.functions import date_format, sum as spark_sum

# Calculate monthly sales revenue
gold_monthly_revenue = gold_sales_df.groupBy(
    date_format(col("bill_date"), "yyyy-MM").alias("sales_month")
).agg(
    spark_sum("line_total_inr").alias("total_revenue_inr")
).orderBy("sales_month")

gold_monthly_revenue.show(100, False)

In [0]:
from pyspark.sql.functions import sum as spark_sum

# Calculate total revenue from all sales
overall_revenue = gold_sales_df.agg(
    spark_sum("line_total_inr").alias("overall_revenue")
)

overall_revenue.show()

In [0]:
# Add all monthly revenue values
monthly_revenue_total = gold_monthly_revenue.agg(
    spark_sum("total_revenue_inr").alias("monthly_revenue_total")
)

monthly_revenue_total.show()

In [0]:
gold_monthly_revenue.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.agri_intelligence.gold_monthly_revenue"
)

In [0]:
# List all tables in the Agri Intelligence schema
spark.sql("SHOW TABLES IN workspace.agri_intelligence").show(100, False)

In [0]:
# Save village-wise business summary
gold_village_summary.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.agri_intelligence.gold_village_summary"
)

In [0]:
spark.table(
    "workspace.agri_intelligence.gold_village_summary"
).show(20, False)

In [0]:
gold_tables = [
    "gold_sales",
    "gold_farmer_summary",
    "gold_village_summary",
    "gold_village_average_bill",
    "gold_village_top_products",
    "gold_monthly_revenue"
]

for table_name in gold_tables:
    row_count = spark.table(
        f"workspace.agri_intelligence.{table_name}"
    ).count()

    print(f"{table_name}: {row_count} rows")

In [0]:
# Keep only sales linked to identified farmers
farmer_purchase_history = gold_sales_df.filter(
    col("farmer_id").isNotNull()
).groupBy(
    "farmer_id",
    "farmer_name",
    "village_name",
    "product_display_name",
    "crop"
).agg(
    spark_sum("quantity_packs").alias("total_packs_purchased"),
    spark_sum("line_total_inr").alias("total_spent_inr")
).orderBy(
    "farmer_name",
    col("total_spent_inr").desc()
)

farmer_purchase_history.show(30, False)

In [0]:
# Save farmer-wise product purchase history
farmer_purchase_history.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.agri_intelligence.gold_farmer_purchase_history"
)

In [0]:
# Count how many separate bills contain each farmer's product
farmer_repeat_purchases = gold_sales_df.filter(
    col("farmer_id").isNotNull()
).groupBy(
    "farmer_id",
    "farmer_name",
    "village_name",
    "product_display_name"
).agg(
    countDistinct("bill_id").alias("purchase_count"),
    spark_sum("quantity_packs").alias("total_packs_purchased"),
    spark_sum("line_total_inr").alias("total_spent_inr")
).filter(
    col("purchase_count") > 1
).orderBy(
    col("purchase_count").desc(),
    col("total_spent_inr").desc()
)

farmer_repeat_purchases.show(30, False)

In [0]:
# Save farmers who purchased the same variety across multiple bills
farmer_repeat_purchases.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.agri_intelligence.gold_farmer_repeat_purchases"
)

In [0]:
from pyspark.sql.functions import when

# Classify farmers by their purchase frequency
farmer_purchase_frequency = gold_farmer_summary.withColumn(
    "purchase_frequency",
    when(col("total_bills") == 1, "One-time")
    .when(col("total_bills").between(2, 3), "Occasional")
    .otherwise("Frequent")
)

farmer_purchase_frequency.select(
    "farmer_id",
    "farmer_name",
    "village_name",
    "total_bills",
    "total_spent_inr",
    "purchase_frequency"
).orderBy(
    col("total_bills").desc()
).show(30, False)

In [0]:
farmer_purchase_frequency.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.agri_intelligence.gold_farmer_purchase_frequency"
)

In [0]:
# Check all Farmer Memory tables
farmer_tables = [
    "gold_farmer_summary",
    "gold_farmer_purchase_history",
    "gold_farmer_repeat_purchases",
    "gold_farmer_purchase_frequency"
]

for table_name in farmer_tables:
    row_count = spark.table(
        f"workspace.agri_intelligence.{table_name}"
    ).count()

    print(f"{table_name}: {row_count} rows")

In [0]:
# Count farmers in each purchase-frequency category
spark.table(
    "workspace.agri_intelligence.gold_farmer_purchase_frequency"
).groupBy(
    "purchase_frequency"
).count().show()

In [0]:
spark.table(
    "workspace.agri_intelligence.gold_farmer_repeat_purchases"
).orderBy(
    col("purchase_count").desc()
).show(20, False)

In [0]:
# Check all Village Pulse tables
village_tables = [
    "gold_village_summary",
    "gold_village_average_bill",
    "gold_village_top_products",
    "gold_monthly_revenue"
]

for table_name in village_tables:
    row_count = spark.table(
        f"workspace.agri_intelligence.{table_name}"
    ).count()

    print(f"{table_name}: {row_count} rows")

In [0]:
# Check the seven villages in the average bill report
spark.table(
    "workspace.agri_intelligence.gold_village_average_bill"
).select(
    "village_name"
).distinct().orderBy(
    "village_name"
).show(20, False)

In [0]:
spark.table(
    "workspace.agri_intelligence.gold_monthly_revenue"
).orderBy(
    "sales_month"
).show(100, False)